In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

df = pd.read_csv('llmtrace_news.csv')
texts = df['text'].dropna().tolist()
print(f"Текстов: {len(texts)}")

Device: cpu
Текстов: 29426


In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
tokenizer.pad_token = tokenizer.eos_token
vocab_size = tokenizer.vocab_size

tokens_list = []
for text in texts:
    tokens_list.extend(tokenizer.encode(text))

print(f"Токенов: {len(tokens_list)}, Vocab: {vocab_size}")

Vocab BPE 5k: 5000
Топ-10 для удаления: {5, 7, 71, 13, 4301, 4304, 18, 19, 55, 27}
Vocab после обрезки и ремаппинга: 4986


In [ ]:
block_size = 64
batch_size = 32

all_tokens = torch.tensor(tokens_list, dtype=torch.long)
n_blocks = len(all_tokens) // (block_size + 1)
X = all_tokens[:n_blocks * (block_size + 1)].view(-1, block_size + 1)
inputs, targets = X[:, :-1], X[:, 1:]
dataset = TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"Блоков: {n_blocks}")

Блоков: 203515


In [ ]:
def train(name, loader, vocab_size, lr, epochs=100):
    config = GPT2Config(
        vocab_size=vocab_size, n_positions=64, n_embd=128, n_layer=2, n_head=2,
        bos_token_id=0, eos_token_id=1,
    )
    model = GPT2LMHeadModel(config).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr) if name=='sgd' else torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        model.train()
        total = 0
        batches = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = model(xb, labels=yb).loss
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
            batches += 1
        avg = total / batches
        history.append(avg)
        if (epoch+1) % 20 == 0:
            print(f"{name} epoch {epoch+1}: loss={avg:.4f}")
    return history

print("=== SGD ===")
loss_sgd = train("sgd", loader, vocab_size, lr=0.5)
np.save("baseline_sgd.npy", np.array(loss_sgd))

print("\n=== Adam ===")
loss_adam = train("adam", loader, vocab_size, lr=0.001)
np.save("baseline_adam.npy", np.array(loss_adam))

plt.plot(loss_sgd, label='SGD')
plt.plot(loss_adam, label='Adam')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.title('Baseline BPE 50k')
plt.savefig("baseline.png", dpi=150)
plt.show()

print(f"SGD final: {loss_sgd[-1]:.4f}, Adam final: {loss_adam[-1]:.4f}, Delta: {loss_sgd[-1]-loss_adam[-1]:.4f}")